# Phase 2: Master Tabular ResNet (Optuna + Entity Embeddings)
This notebook trains the ultimate Neural Network using:
1. **Entity Embeddings:** For categorical features.
2. **Label Smoothing:** To heavily optimize the LogLoss metric.
3. **Optuna:** To mathematically find the perfect hyperparameters.
4. **Stratified 5-Fold CV:** For robust OOF evaluation.

In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import log_loss, roc_auc_score
import optuna
import os
import copy
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


/Users/USER/Desktop/zindi/ml_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# 1. Load Data & Create Pipeline
print("Loading raw domain features...")
train_df = pd.read_parquet('../data/features/train_features.parquet', engine='fastparquet')
test_df = pd.read_parquet('../data/features/test_features.parquet', engine='fastparquet')

target_col = 'liquidity_stress_next_30d'
y = train_df[target_col].astype(np.float32).values
train_df = train_df.drop(columns=[target_col, 'ID'], errors='ignore')

test_ids = test_df['ID'].values if 'ID' in test_df.columns else np.arange(len(test_df))
test_df = test_df.drop(columns=['ID'], errors='ignore')

# Identify categorical vs continuous columns
cat_cols = [c for c in train_df.columns if train_df[c].dtype == 'object' or train_df[c].nunique() < 10]
cont_cols = [c for c in train_df.columns if c not in cat_cols]

print(f"Found {len(cat_cols)} categorical and {len(cont_cols)} continuous features.")

Loading raw domain features...
Found 30 categorical and 198 continuous features.


In [15]:
# Preprocessing Pipeline
def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))

# 1. Log Transform Skewed Continuous Features
skewed_cols = [c for c in cont_cols if any(kw in c for kw in ['amount', 'value', 'volume', 'bal', 'flow', 'runway', 'peak', 'shock', 'volatility'])]
for col in skewed_cols:
    train_df[col] = signed_log1p(train_df[col])
    test_df[col] = signed_log1p(test_df[col])

# 2. Fill NaNs
train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

# 3. Label Encode Categoricals
cat_dims = []
cat_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    # Handle unseen categories in test safely
    test_df[col] = test_df[col].astype(str).map(lambda s: '<unknown>' if s not in le.classes_ else s)
    le.classes_ = np.append(le.classes_, '<unknown>')
    test_df[col] = le.transform(test_df[col])
    cat_dims.append(len(le.classes_))

# 4. Standard Scale Continuous
scaler = StandardScaler()
train_df[cont_cols] = scaler.fit_transform(train_df[cont_cols])
test_df[cont_cols] = scaler.transform(test_df[cont_cols])

# Convert to numpy arrays
X_cat = train_df[cat_cols].values.astype(np.int64)
X_cont = train_df[cont_cols].values.astype(np.float32)
X_test_cat = test_df[cat_cols].values.astype(np.int64)
X_test_cont = test_df[cont_cols].values.astype(np.float32)

In [16]:
# PyTorch Dataset
class LiquidityDataset(Dataset):
    def __init__(self, X_cat, X_cont, y=None, apply_smoothing=False):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_cont = torch.tensor(X_cont, dtype=torch.float32)
        
        # Label Smoothing (0 -> 0.02, 1 -> 0.98)
        if y is not None:
            y = np.array(y)
            if apply_smoothing:
                y = np.where(y == 1, 0.98, 0.02)
            self.y = torch.tensor(y, dtype=torch.float32)
        else:
            self.y = None
        
    def __len__(self):
        return len(self.X_cont)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_cat[idx], self.X_cont[idx], self.y[idx]
        return self.X_cat[idx], self.X_cont[idx]

In [17]:
# Model Architecture: Tabular ResNet with Entity Embeddings
class ResNetBlock(nn.Module):
    def __init__(self, hidden_dim, dropout_rate):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.act = nn.GELU()
        
    def forward(self, x):
        return self.act(x + self.layer(x))

class TabularResNet(nn.Module):
    def __init__(self, cat_dims, num_cont, hidden_dim=256, num_blocks=3, dropout_rate=0.2):
        super().__init__()
        
        # Entity Embeddings
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, min(50, (num_classes + 1) // 2))
            for num_classes in cat_dims
        ])
        
        total_embed_dim = sum(e.embedding_dim for e in self.embeddings) if self.embeddings else 0
        input_dim = total_embed_dim + num_cont
        
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
        
        self.blocks = nn.ModuleList([
            ResNetBlock(hidden_dim, dropout_rate) for _ in range(num_blocks)
        ])
        
        self.output_layer = nn.Linear(hidden_dim, 1)
        
    def forward(self, x_cat, x_cont):
        if len(self.embeddings) > 0:
            x_emb = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
            x_emb = torch.cat(x_emb, dim=1)
            x = torch.cat([x_emb, x_cont], dim=1)
        else:
            x = x_cont
            
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x).squeeze()

In [18]:
# Training Function
def train_model(X_cat_t, X_cont_t, y_t, X_cat_v, X_cont_v, y_v, params):
    train_dataset = LiquidityDataset(X_cat_t, X_cont_t, y_t, apply_smoothing=True)
    val_dataset = LiquidityDataset(X_cat_v, X_cont_v, y_v, apply_smoothing=False) # No smoothing for pure validation loss
    
    train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=params['batch_size'] * 2, shuffle=False)
    
    model = TabularResNet(
        cat_dims=cat_dims, num_cont=len(cont_cols),
        hidden_dim=params['hidden_dim'], num_blocks=params['num_blocks'], dropout_rate=params['dropout_rate']
    ).to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    
    best_val_loss = float('inf')
    best_weights = None
    patience_counter = 0
    
    for epoch in range(params['epochs']):
        model.train()
        for b_cat, b_cont, b_y in train_loader:
            b_cat, b_cont, b_y = b_cat.to(device), b_cont.to(device), b_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(b_cat, b_cont), b_y)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        val_loss, val_preds, val_targets = 0, [], []
        with torch.no_grad():
            for b_cat, b_cont, b_y in val_loader:
                b_cat, b_cont, b_y = b_cat.to(device), b_cont.to(device), b_y.to(device)
                outputs = model(b_cat, b_cont)
                val_loss += criterion(outputs, b_y).item() * b_cont.size(0)
                val_preds.extend(torch.sigmoid(outputs).cpu().numpy())
                val_targets.extend(b_y.cpu().numpy())
                
        val_loss /= len(val_dataset)
        scheduler.step(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= params['early_stopping']:
                break
                
    model.load_state_dict(best_weights)
    return model, best_val_loss, val_preds

In [19]:
# Optuna Hyperparameter Optimization
def objective(trial):
    # Sample parameters
    params = {
        'hidden_dim': trial.suggest_categorical('hidden_dim', [128, 256, 512]),
        'num_blocks': trial.suggest_int('num_blocks', 1, 4),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        'batch_size': 1024,
        'epochs': 30,
        'early_stopping': 5
    }
    
    # Fast single 80/20 split for Optuna
    Xc_train, Xc_val, Xco_train, Xco_val, y_train, y_val = train_test_split(
        X_cat, X_cont, y, test_size=0.2, stratify=y, random_state=42
    )
    
    _, val_loss, _ = train_model(Xc_train, Xco_train, y_train, Xc_val, Xco_val, y_val, params)
    return val_loss

print("Starting Optuna Optimization (20 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)

best_params = study.best_params
best_params['batch_size'] = 1024
best_params['epochs'] = 100
best_params['early_stopping'] = 10
print("\nBest Params Found:", best_params)

[I 2026-09-03 18:33:38,282] A new study created in memory with name: no-name-60fabb6e-3920-4465-ab8f-0a3ecf3c2204


Starting Optuna Optimization (20 trials)...


[I 2026-09-03 18:33:52,919] Trial 0 finished with value: 0.32539966773986817 and parameters: {'hidden_dim': 256, 'num_blocks': 2, 'dropout_rate': 0.3297665738204213, 'lr': 0.00014260097828328638, 'weight_decay': 1.008465112929754e-05}. Best is trial 0 with value: 0.32539966773986817.
[I 2026-09-03 18:34:09,225] Trial 1 finished with value: 0.3243987636566162 and parameters: {'hidden_dim': 512, 'num_blocks': 3, 'dropout_rate': 0.4105448248307124, 'lr': 0.00030326216119181597, 'weight_decay': 0.002984068182954751}. Best is trial 1 with value: 0.3243987636566162.
[I 2026-09-03 18:34:26,069] Trial 2 finished with value: 0.319925717830658 and parameters: {'hidden_dim': 128, 'num_blocks': 3, 'dropout_rate': 0.25830587606173244, 'lr': 0.00018473455136907053, 'weight_decay': 0.0003796251696083204}. Best is trial 2 with value: 0.319925717830658.
[I 2026-09-03 18:34:34,436] Trial 3 finished with value: 0.31382034182548524 and parameters: {'hidden_dim': 256, 'num_blocks': 1, 'dropout_rate': 0.392


Best Params Found: {'hidden_dim': 256, 'num_blocks': 2, 'dropout_rate': 0.4992062513355332, 'lr': 0.002809496583849211, 'weight_decay': 0.0009755999311034751, 'batch_size': 1024, 'epochs': 100, 'early_stopping': 10}


In [20]:
# Final 5-Fold CV using Best Params
print("\nStarting 5-Fold Stratified CV with Best Params...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(y))
fold_losses = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_cont, y)):
    print(f"\n--- Fold {fold+1} ---")
    model, best_val_loss, val_preds = train_model(
        X_cat[train_idx], X_cont[train_idx], y[train_idx],
        X_cat[val_idx], X_cont[val_idx], y[val_idx],
        best_params
    )
    oof_preds[val_idx] = val_preds
    fold_losses.append(best_val_loss)
    print(f"Fold {fold+1} LogLoss: {best_val_loss:.4f}")
    
print(f"\nFinal CV Mean LogLoss: {np.mean(fold_losses):.4f}")
print(f"Final OOF ROC-AUC: {roc_auc_score(y, oof_preds):.4f}")


Starting 5-Fold Stratified CV with Best Params...

--- Fold 1 ---
Fold 1 LogLoss: 0.2908

--- Fold 2 ---
Fold 2 LogLoss: 0.2988

--- Fold 3 ---
Fold 3 LogLoss: 0.2990

--- Fold 4 ---
Fold 4 LogLoss: 0.3000

--- Fold 5 ---
Fold 5 LogLoss: 0.3059

Final CV Mean LogLoss: 0.2989
Final OOF ROC-AUC: 0.8512


In [21]:
print("Saving PyTorch NN OOF predictions for Ensembling...")
np.save("../data/features/nn_oof_preds.npy", oof_preds)
print("Saved to data/features/nn_oof_preds.npy!")

Saving PyTorch NN OOF predictions for Ensembling...
Saved to data/features/nn_oof_preds.npy!
